## Task 1:
Create a bronze table that ingests raw sales data as-is (no transformations), preserving all original
columns plus an ingestion timestamp.

In [0]:
%sql
-- This table was made by the pipeline in the file ingestion_to_bronze
select * from cyntexa_dev.bronze.bronze_sales

## Task 2:
Build a silver table from bronze that removes duplicates, fixes data types, and drops clearly invalid
rows.

In [0]:
%sql
-- This table was made by the pipeline in the file bronze_to_silver
select * from cyntexa_dev.silver.silver_sales

## Task 3: 
Build a gold table that aggregates silver into a business-ready view (e.g., daily revenue by store).

In [0]:
%sql
-- This table was made by the pipeline in the file silver_to_gold
select * from cyntexa_dev.gold.daily_revenue_by_store

## Task 4:
Diagram your bronze/silver/gold pipeline and label, for each layer, who the primary consumer is
(engineers, analysts, executives).

![](/Volumes/cyntexa_dev/bronze/raw/images/Screenshot 2026-08-26 122225.png)


The label for the primary consumers of each table:
- **Bronze(bronze_sales)** -> primary consumers are data engineers (This layer contains the raw, unrefined data which is then processed to make it business ready)
- **Silver(silver_sales)** -> primary consumers are data engineers and analysts (clean, typed and deduplicated, analysts can query this data but it is still not business ready)
- **Gold(daily_revenue_by_store)** -> primary consumers are analysts and executives (aggergated, business ready, dashboard ready)

## Task 5:
Recreate one part of your silver transformation using Lakeflow Designer's visual, no-code interface
and compare the experience to writing it in code.

![](/Volumes/cyntexa_dev/bronze/raw/images/Screenshot 2026-08-26 134317.png)

Comparing the expirence between code and no code:
- **Speed to build:**
    - For the code it was first if the syntax is know
    - Lakeflow designer is faster as I have to only add the nodes which is required without knowing the internal working of the spark.
- **Errors:**
    - Errors are only surfaced when the pipeline run only, I made a type during the cast which created a error `'Column' object is not callable` which I came across only after the pipeline run.
    - Each node validates its own configuration inline, so the erros are instant and a typo or syntax errors are very less likely to occour.
- **Flexiblity:**
    - I have full control in the pyspark.
    - I am bound by the actions the node types exposes, which is fine for filter and casts but multiline logic may be difficult.

## Task 6:
Chain bronze → silver → gold as a Lakeflow Job with proper task dependencies, and configure it to
run on a schedule.

![](/Volumes/cyntexa_dev/bronze/raw/images/pipeline_run.png)

A Lakeflow Job was created to orchestrate the bronze → silver → gold pipeline on a recurring schedule. The Job consists of two tasks with an explicit dependency between them:
- **Task 1 —** run_sales_pipeline (type: Pipeline): triggers the existing Lakeflow pipeline covering bronze_sales → stg_sales → silver_sales → daily_revenue_by_store.
- **Task 2 —** notify_pipeline_complete (type: Notebook): depends on Task 1 succeeding. It queries daily_revenue_by_store for a row count and raises an exception if the table is empty, which causes this task — and therefore the Job run — to fail if the pipeline silently produced no output. This makes the dependency meaningful

Jobs YAML file
```
resources:
  jobs:
    Day_6_Task_6_job:
      name: Day_6_Task_6_job
      schedule:
        quartz_cron_expression: 29 0 6 * * ?
        timezone_id: Asia/Calcutta
        pause_status: UNPAUSED
      tasks:
        - task_key: run_sales_pipeline
          pipeline_task:
            pipeline_id: 9f26ea8d-57cb-4f44-9f7a-94881ebe290d
        - task_key: notify_pipeline_completion
          depends_on:
            - task_key: run_sales_pipeline
          notebook_task:
            notebook_path: /Workspace/Users/kiritjain@cyntexa.com/Data-Engineering-Assignment/Day_6/notifying_notebook_task_6
            source: WORKSPACE
          min_retry_interval_millis: 900000
          disable_auto_optimization: true
      queue:
        enabled: true
      performance_target: PERFORMANCE_OPTIMIZED

```

## Task 7:
Extend the gold layer with a second aggregation for a different stakeholder (e.g., the inventory team)
and justify why it belongs in gold rather than being computed ad hoc by that team.

In [0]:
%sql
-- This table was made by the pipeline in the file silver_to_gold
select * from cyntexa_dev.gold.daily_units_by_product

This aggergation belongs in the gold layer as it answer a recurring and well defined business question that a second stakeholder team needs repeatedly, defining it once in the gold keeps it away from duplicates and dashboard querries as fast, and it makes sure that sales adn inventory teams are using the same goverened body rather than independently derived one.

## Task 8:
Write a short design note on which parts of this pipeline should run in the customer's data plane vs.
rely on Databricks' control plane, and what that means for a network/security review.

### What runs in the control plane
- Lakeflow pipeline orchestration/DAG scheduling (bronze→silver→gold ordering)
- Lakeflow Job scheduler (daily trigger from Task 6)
- Lakeflow Designer UI (Task 5's visual canvas)
- Notebook source code, pipeline configuration metadata

### What runs in the data plane
- Unity Catalog Volume storing raw sales CSVs
- All Delta tables: bronze_sales, silver_sales, daily_revenue_by_store, daily_units_by_product
- Spark cluster compute executing ingestion, CDC deduplicate, and aggregation logic
- The actual sales data itself (customer_id, order amounts, etc.)

### Why the split matters here
The Split matter because the sensitive data (customer_id, customer name or contact info) should never leave the company's cloud as it maybe prone to leaks.

Databrick's control plane only needs to run the notebooks and pipeline, and it only ever see the metadata

### Network/security review implications
Our security team needs to approve network connectivity between our
cloud account (data plane) and Databricks' control plane,
permissions letting Databricks orchestrate cluster startup without direct data
access, and data residency — our Volume/storage stays in our chosen cloud region.

## Task 9:
(Data Analyst) Build a query or lightweight dashboard directly against the gold table, and identify
one data-quality issue you can trace back to a specific bronze or silver transformation decision.

In [0]:
%sql
-- 1. Prove the pipeline ran clean despite bad input
SELECT COUNT(*) AS total_rows,
       COUNT(order_date) AS non_null_dates,
       COUNT(*) - COUNT(order_date) AS null_dates,
       COUNT(total_amount) AS non_null_amounts,
       COUNT(*) - COUNT(total_amount) AS null_amounts
FROM cyntexa_dev.silver.silver_sales;

In [0]:
%sql
-- 2. Pull out the exact rows where casting silently failed
SELECT order_id, customer_id, product_id, quantity,
       discount_amount, total_amount, order_date
FROM cyntexa_dev.silver.silver_sales
WHERE order_date IS NULL OR total_amount IS NULL OR discount_amount IS NULL
ORDER BY order_id;

In [0]:
%sql
-- 3. Confirm row-count gap between bronze and silver
SELECT
  (SELECT COUNT(*) FROM cyntexa_dev.bronze.bronze_sales) AS bronze_count,
  (SELECT COUNT(*) FROM cyntexa_dev.silver.silver_sales) AS silver_count;

### Query against gold

```
-- 1. Prove the pipeline ran clean despite bad input
SELECT COUNT(*) AS total_rows,
       COUNT(order_date) AS non_null_dates,
       COUNT(*) - COUNT(order_date) AS null_dates,
       COUNT(total_amount) AS non_null_amounts,
       COUNT(*) - COUNT(total_amount) AS null_amounts
FROM cyntexa_dev.silver.silver_sales;
```
![](/Volumes/cyntexa_dev/bronze/raw/images/query1.png)

```
-- 2. Pull out the exact rows where casting silently failed
SELECT order_id, customer_id, product_id, quantity,
       discount_amount, total_amount, order_date
FROM cyntexa_dev.silver.silver_sales
WHERE order_date IS NULL OR total_amount IS NULL OR discount_amount IS NULL
ORDER BY order_id;
```
![](/Volumes/cyntexa_dev/bronze/raw/images/query2.png)

```
-- 3. Confirm row-count gap between bronze and silver
SELECT
  (SELECT COUNT(*) FROM cyntexa_dev.bronze.bronze_sales) AS bronze_count,
  (SELECT COUNT(*) FROM cyntexa_dev.silver.silver_sales) AS silver_count;

```

![](/Volumes/cyntexa_dev/bronze/raw/images/query3.png)

### Data-quality issues identified

**1. Missing store dimension (modeling gap)**
daily_revenue_by_store does not actually break revenue down by store — it
reports a single daily total across all stores. Tracing back through the
pipeline: bronze_sales ingests the raw CSV as-is, and the source data itself
never included a store_id column. Since silver only cleans and casts existing
columns rather than deriving new ones, and gold can only group by what silver
provides, the missing dimension propagated silently from bronze all the way
to the final table name, which now overstates what the data actually contains.
The pipeline never errored, because a missing dimension isn't a validation
failure — it's a semantic gap only visible to someone actually querying gold.

**2. Silent nulling of malformed values during casting (technical gap)**
Silver's cleaning logic relies entirely on .cast(DateType()) and
.cast(DoubleType()) to type-correct order_date, total_amount, and
discount_amount. These casts do not error on malformed input — they silently
return null:

- A date like "25/08/2026" (non-ISO format) or "Aug 25, 2026" casts to null,
  since Spark's default date cast only recognizes yyyy-MM-dd.
- A value like "$45.00" or "1,200.50" (currency symbols or thousands
  separators) casts to null, since Spark's numeric cast cannot strip
  non-numeric characters.

Unlike quantity, which is indirectly protected because the existing
expect_or_drop("valid_quantity", "quantity > 0") rule drops rows where the
cast already failed (null > 0 evaluates to false), order_date, total_amount,
and discount_amount have no equivalent null check. A row with a malformed
date or a currency-formatted amount is never flagged or dropped — it silently
carries a null into gold, where it either gets excluded from date_trunc-based
grouping or quietly deflates SUM(total_amount) without any error, warning, or
row count discrepancy to signal that anything went wrong.

**3. No normalization for whitespace or case**
Text fields like product_id and customer_id are never trimmed or
case-normalized in silver. If the same product ID appears as "ABC-99" and
" ABC-99 " (or in inconsistent case) across different source files, gold's
groupBy("product_id") would treat these as distinct products, silently
fragmenting the daily_units_by_product aggregation rather than merging them.

### Why this matters
All three issues share a common root cause: silver's current validation
(expect_or_drop) only covers quantity and order_id explicitly. Every other
column is cast without a corresponding null/format check, so bad data in
those columns doesn't fail loudly — it just degrades gold's numbers quietly.
The fix isn't a single line change; it requires either stricter casting
(explicit format strings for dates, regex cleanup for currency strings before
casting) or additional expect_or_drop rules on the currently-unvalidated
columns, plus, for the store_id case, a change to what bronze ingests in the
first place — silver and gold can only be as complete as what bronze receives.